# Static ASL Recognition — End-to-End Colab Notebook

Chạy các cell theo thứ tự để tải dataset từ Hugging Face, cắt bàn tay bằng MediaPipe, train MobileNetV2 và đánh giá 36 lớp. Notebook tự chứa này là entry point chính của đồ án.

In [ ]:
%pip -q install 'mediapipe>=0.10.14' 'huggingface-hub>=0.25' pandas scikit-learn matplotlib seaborn pyyaml
# Google Colab đã có TensorFlow GPU; không cài lại TensorFlow để tránh xung đột tensorflow-text.

In [ ]:
from pathlib import Path
import json, random, shutil, urllib.request, zipfile
from datetime import datetime, timezone
import cv2, mediapipe as mp, numpy as np, pandas as pd, tensorflow as tf
from huggingface_hub import snapshot_download
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ID = 'hnam25/asl-hand-gesture-images'
REVISION = None  # Thay bằng commit SHA sau khi dataset ổn định để tái lập kết quả.
ROOT = Path('/content/asl-static-recognition') if Path('/content').exists() else Path.cwd() / 'asl-static-recognition'
RAW, PROCESSED = ROOT / 'data/raw', ROOT / 'data/processed'
METADATA, SPLITS = ROOT / 'data/metadata', ROOT / 'data/splits'
MODELS, METRICS, FIGURES, LOGS = (ROOT / 'outputs/models', ROOT / 'outputs/metrics', ROOT / 'outputs/figures', ROOT / 'outputs/logs')
for directory in (RAW, PROCESSED, METADATA, SPLITS, MODELS, METRICS, FIGURES, LOGS, ROOT / 'assets'):
    directory.mkdir(parents=True, exist_ok=True)
DEBUG_LOG = LOGS / 'cli_debug.log'
def debug(stage, **fields):
    message = f"{datetime.now(timezone.utc).isoformat()} | {stage}" + (f" | {json.dumps(fields, ensure_ascii=False)}" if fields else '')
    with DEBUG_LOG.open('a', encoding='utf-8') as handle: handle.write(message + '\n')
    print(message, flush=True)

CLASSES = [str(i) for i in range(10)] + [chr(i) for i in range(ord('A'), ord('Z') + 1)]
SEED, IMAGE_SIZE, BATCH_SIZE, EPOCHS = 42, 224, 32, 20
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
debug('environment_ready', tensorflow=tf.__version__, gpus=[device.name for device in tf.config.list_physical_devices('GPU')])
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
print('Project:', ROOT)

In [ ]:
# Chỉ tải archive raw (không tải archive processed), sau đó giải nén an toàn.
debug('dataset_download_start', repo_id=REPO_ID, revision=REVISION)
snapshot_download(repo_id=REPO_ID, repo_type='dataset', revision=REVISION, local_dir=RAW, allow_patterns='**/ASL_Raw_Images.zip')
archives = list(RAW.rglob('ASL_Raw_Images.zip'))
if not archives: raise RuntimeError(f'Không tìm thấy ASL_Raw_Images.zip trong {REPO_ID}.')
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        destination = (RAW / member.filename).resolve()
        if RAW.resolve() not in destination.parents and destination != RAW.resolve(): raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
    archive.extractall(RAW)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
# Archive có thể chứa một thư mục cha; tự tìm directory có đủ 36 folder class.
class_roots = [p.parent for p in RAW.rglob('0') if p.is_dir() and all((p.parent / label).is_dir() for label in CLASSES)]
if not class_roots: raise RuntimeError('Không tìm thấy layout 0-9/A-Z sau khi giải nén archive.')
CLASS_ROOT = sorted(class_roots, key=lambda p: len(p.parts))[0]
images = [p for p in CLASS_ROOT.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS]
if not images: raise RuntimeError('Archive đã giải nén nhưng không có ảnh được hỗ trợ.')
debug('dataset_ready', images=len(images), class_root=str(CLASS_ROOT))
print(f'Downloaded/extracted {len(images):,} images from {REPO_ID}; class root: {CLASS_ROOT}')
json.dump(CLASSES, (METADATA / 'class_names.json').open('w'), indent=2)

In [ ]:
# Audit ảnh nguồn; cache được tái sử dụng khi archive/dataset không thay đổi.
import hashlib
audit_path = METADATA / 'dataset_audit.csv'; debug('audit_start', cache_exists=audit_path.exists())
if audit_path.exists():
    audit = pd.read_csv(audit_path); print('Using cached dataset audit:', audit_path)
else:
    audit_rows = []
    for label in CLASSES:
        for path in sorted((CLASS_ROOT / label).rglob('*')):
            if path.suffix.lower() not in IMAGE_EXTENSIONS: continue
            image = cv2.imread(str(path))
            digest = hashlib.sha256(path.read_bytes()).hexdigest() if image is not None else ''
            audit_rows.append({'label': label, 'path': str(path), 'status': 'ok' if image is not None else 'unreadable', 'sha256': digest})
    audit = pd.DataFrame(audit_rows); audit.to_csv(audit_path, index=False)
    audit[audit.sha256.duplicated(False) & (audit.sha256 != '')].to_csv(METADATA / 'duplicate_images.csv', index=False)
class_counts = audit[audit.status == 'ok'].groupby('label').size().reindex(CLASSES, fill_value=0)
debug('audit_complete', readable=int((audit.status == 'ok').sum()), unreadable=int((audit.status != 'ok').sum()), min_per_class=int(class_counts.min()), max_per_class=int(class_counts.max()))
print('Images per class:', class_counts.to_dict(), flush=True)
assert audit[audit.status == 'ok'].groupby('label').size().reindex(CLASSES, fill_value=0).min() >= 10, 'Mỗi lớp cần ít nhất 10 ảnh để split stratified.'

In [ ]:
# MediaPipe Hand Landmarker: download model asset một lần và crop vùng tay có padding.
debug('segmentation_start', images=int((audit.status == 'ok').sum()))
TASK_MODEL = ROOT / 'assets/hand_landmarker.task'
if not TASK_MODEL.exists():
    urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task', TASK_MODEL)
options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(TASK_MODEL)),
    running_mode=mp.tasks.vision.RunningMode.IMAGE, num_hands=1,
    min_hand_detection_confidence=.5, min_hand_presence_confidence=.5, min_tracking_confidence=.5)
landmarker = mp.tasks.vision.HandLandmarker.create_from_options(options)
failures = []
try:
    for row in audit[audit.status == 'ok'].itertuples():
        image = cv2.imread(row.path); rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        result = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
        destination = PROCESSED / row.label / Path(row.path).name
        status = 'ok'
        if not result.hand_landmarks:
            status = 'no_hand_detected'
        else:
            h, w = image.shape[:2]; pts = result.hand_landmarks[0]
            xs, ys = [p.x*w for p in pts], [p.y*h for p in pts]; side = max(max(xs)-min(xs), max(ys)-min(ys)) * 1.36
            cx, cy = (min(xs)+max(xs))/2, (min(ys)+max(ys))/2
            x1, y1, x2, y2 = max(0,int(cx-side/2)), max(0,int(cy-side/2)), min(w,int(cx+side/2)), min(h,int(cy+side/2))
            if x2 <= x1 or y2 <= y1: status = 'invalid_bbox'
            else:
                destination.parent.mkdir(parents=True, exist_ok=True); cv2.imwrite(str(destination), image[y1:y2, x1:x2])
        failures.append({'label': row.label, 'source_path': row.path, 'processed_path': str(destination), 'status': status})
        if len(failures) % 1000 == 0: debug('segmentation_progress', processed=len(failures))
finally:
    landmarker.close()
segmentation = pd.DataFrame(failures); segmentation.to_csv(METADATA / 'segmentation_failures.csv', index=False)
debug('segmentation_complete', results=segmentation.status.value_counts().to_dict())
print('Segmentation status:', segmentation.status.value_counts().to_dict(), flush=True)

In [ ]:
# Deduplicate exact raw-image copies, then create a locked 80/10/10 split.
debug('split_start')
audit_hashes = audit[audit.status == 'ok'][['path', 'label', 'sha256']].copy()
usable = segmentation[segmentation.status == 'ok'].merge(audit_hashes, how='left', left_on='source_path', right_on='path', suffixes=('', '_audit'), validate='one_to_one')
if usable.sha256.isna().any() or (usable.label != usable.label_audit).any(): raise RuntimeError('Audit and segmentation manifests do not match.')
if (usable.groupby('sha256').label.nunique() > 1).any(): raise RuntimeError('The same raw image hash has conflicting labels.')
usable = usable.sort_values(['sha256', 'source_path'], kind='stable').reset_index(drop=True)
usable['duplicate_group_size'] = usable.groupby('sha256').sha256.transform('size')
usable['canonical_source_path'] = usable.groupby('sha256').source_path.transform('first')
usable['is_canonical'] = usable.source_path.eq(usable.canonical_source_path)
usable[['label', 'source_path', 'processed_path', 'sha256', 'duplicate_group_size', 'canonical_source_path', 'is_canonical']].to_csv(METADATA / 'deduplication_manifest.csv', index=False)
before_dedup = len(usable); usable = usable[usable.is_canonical].copy()
if usable.sha256.duplicated().any(): raise RuntimeError('Exact deduplication failed.')
train, holdout = train_test_split(usable, train_size=.8, stratify=usable.label, random_state=SEED)
val, test = train_test_split(holdout, train_size=.5, stratify=holdout.label, random_state=SEED)
for name, frame in {'train': train, 'val': val, 'test': test}.items(): frame[['processed_path', 'label']].to_csv(SPLITS / f'{name}.csv', index=False)
if set(train.sha256) & set(val.sha256) or set(train.sha256) & set(test.sha256) or set(val.sha256) & set(test.sha256): raise RuntimeError('Hash leakage detected.')
json.dump({'policy': 'one representative per exact SHA-256', 'before_deduplication': before_dedup, 'after_deduplication': len(usable), 'removed_exact_duplicates': before_dedup-len(usable), 'seed': SEED}, (METADATA / 'split_manifest.json').open('w'), indent=2)
label_to_index = {label: i for i, label in enumerate(CLASSES)}
def make_ds(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame.processed_path.values, frame.label.map(label_to_index).values))
    if training: ds = ds.shuffle(len(frame), seed=SEED)
    def load(path, label):
        x = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False); x.set_shape([None, None, 3])
        return tf.image.resize(tf.cast(x, tf.float32), (IMAGE_SIZE, IMAGE_SIZE)), tf.one_hot(label, len(CLASSES))
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
train_ds, val_ds, test_ds = make_ds(train, True), make_ds(val), make_ds(test)
debug('split_complete', before_dedup=before_dedup, after_dedup=len(usable), train=len(train), validation=len(val), test=len(test))
print({'before_dedup': before_dedup, 'after_dedup': len(usable), 'train': len(train), 'validation': len(val), 'test': len(test)}, flush=True)


In [ ]:
# MobileNetV2 baseline: frozen ImageNet backbone + 36-class head.
debug('training_start', epochs=EPOCHS, batch_size=BATCH_SIZE)
backbone = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
backbone.trainable = False
inputs = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3), name='image')
x = backbone(tf.keras.applications.mobilenet_v2.preprocess_input(inputs), training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x); x = tf.keras.layers.Dropout(.2)(x)
model = tf.keras.Model(inputs, tf.keras.layers.Dense(len(CLASSES), activation='softmax', name='classification')(x))
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
checkpoint = MODELS / 'baseline_mobilenetv2.keras'
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2, callbacks=[
    tf.keras.callbacks.ModelCheckpoint(checkpoint, monitor='val_accuracy', save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=.2),
])
pd.DataFrame(history.history).to_csv(LOGS / 'baseline_history.csv', index=False)
debug('training_complete', best_val_accuracy=float(max(history.history['val_accuracy'])))

In [ ]:
# Test metrics, confusion matrix và phân tích bắt buộc O/0.
debug('evaluation_start')
best = tf.keras.models.load_model(checkpoint); probabilities = best.predict(test_ds, verbose=1); y_pred = probabilities.argmax(1)
y_true = test.label.map(label_to_index).to_numpy()
report = classification_report(y_true, y_pred, labels=range(36), target_names=CLASSES, output_dict=True, zero_division=0)
matrix = confusion_matrix(y_true, y_pred, labels=range(36))
pd.DataFrame(report).T.to_csv(METRICS / 'classification_report.csv')
pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).to_csv(METRICS / 'confusion_matrix.csv')
o, zero = label_to_index['O'], label_to_index['0']
o_zero = {'test_accuracy': float(accuracy_score(y_true, y_pred)), 'O_recall': report['O']['recall'], '0_recall': report['0']['recall'], 'O_to_0': int(matrix[o, zero]), '0_to_O': int(matrix[zero, o])}
json.dump(o_zero, (METRICS / 'o_zero_analysis.json').open('w'), indent=2); debug('evaluation_complete', **o_zero); print(o_zero, flush=True)
plt.figure(figsize=(16,13)); sns.heatmap(matrix, cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES); plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.savefig(FIGURES / 'confusion_matrix.png', dpi=180); plt.close()
if o_zero['O_recall'] < .95 or o_zero['0_recall'] < .95: print('Chưa đạt mục tiêu O/0: chuyển sang fine-tuning/augmentation, vẫn giữ nguyên test split.')

## Realtime local

Webcam không chạy ổn định trên Colab headless. Tải `outputs/models/baseline_mobilenetv2.keras` về máy local và chạy `python -m src.inference.webcam_demo`. Luôn dừng Colab runtime sau khi tải artifacts: `colab stop -s asl-training`.